# 04 · RAG con LangChain

**Bloque:** 1:10–1:35 · **Práctica:** 1:19–1:35

### Teoría breve
- **RAG (Retrieval-Augmented Generation / Generación Aumentada por Recuperación):** recupera contexto relevante **antes** de responder.
- **Pipeline:** documentos → *chunks* (fragmentos) → *embeddings* → *vector store* → *retriever* → LLM.

Indexaremos `data/agentes_ia.md` y haremos preguntas sobre él.

In [ ]:
import sys, os
REPO = os.path.abspath("..")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from config import get_chat_model, get_embeddings
print("Entorno cargado ✔")

## Paso 1 · Cargar y trocear el documento

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

texto = open(os.path.join(REPO, "data", "agentes_ia.md"), encoding="utf-8").read()
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.create_documents([texto])
print("Fragmentos generados:", len(chunks))

## Paso 2 · Embeddings + vector store
> **TODO:** crea el índice FAISS a partir de los `chunks` y los `emb` con `FAISS.from_documents`.

In [ ]:
from langchain_community.vectorstores import FAISS

emb = get_embeddings()
# TODO: crea el vector store FAISS
vs = ____
retriever = vs.as_retriever(search_kwargs={"k": 3})
print("Índice vectorial listo ✔")

## Paso 3 · La cadena RAG
Recuperamos contexto y se lo damos al LLM junto con la pregunta.

> **TODO:** invoca `prompt | llm` con `contexto` y `pregunta`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

llm = get_chat_model()
prompt = ChatPromptTemplate.from_template(
    "Responde la pregunta usando SOLO el contexto. Si no está, di que no lo sabes.\n\n"
    "Contexto:\n{contexto}\n\nPregunta: {pregunta}"
)

def responder_rag(pregunta: str):
    docs = retriever.invoke(pregunta)
    contexto = "\n\n".join(d.page_content for d in docs)
    # TODO: invoca la cadena prompt|llm con contexto y pregunta
    msg = ____
    return msg.content, docs

## ✅ Checkpoint 4
Haz una pregunta y verifica que la respuesta **cita el contexto** recuperado.

In [ ]:
resp, fuentes = responder_rag("¿Qué es MCP y quién lo creó?")
print(resp)
print("\n--- Fragmentos usados como fuente ---")
for d in fuentes:
    print("•", d.page_content[:90].replace("\n", " "), "...")